In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

print("All imports successful!")

All imports successful!


In [9]:
df = pd.read_csv("https://raw.githubusercontent.com/The1OGShaggy/pyMAISE_VarianceReductionBiasMitigationFinal/refs/heads/main/chf_combined.csv")
df_lhs = pd.read_csv("https://raw.githubusercontent.com/The1OGShaggy/pyMAISE_VarianceReductionBiasMitigationFinal/refs/heads/main/lhs_samples.csv")
df_multi_stratified = pd.read_csv("https://raw.githubusercontent.com/The1OGShaggy/pyMAISE_VarianceReductionBiasMitigationFinal/refs/heads/main/multi_stratified_samples.csv")
df_random_samples = pd.read_csv("https://raw.githubusercontent.com/The1OGShaggy/pyMAISE_VarianceReductionBiasMitigationFinal/refs/heads/main/random_samples.csv")

#check that all datasets loaded properly
print("Combined dataset shape:", df.shape)
print("LHS samples shape:", df_lhs.shape)
print("Multi-stratified samples shape:", df_multi_stratified.shape)
print("Random samples shape:", df_random_samples.shape)
print("All datasets loaded successfully!")

Combined dataset shape: (2500, 7)
LHS samples shape: (1000, 7)
Multi-stratified samples shape: (1267, 7)
Random samples shape: (1000, 7)
All datasets loaded successfully!


In [10]:
#pick target
target = "CHF (kW m-2)"

features = [col for col in df.columns if col != target]

In [11]:
#non sampled dataset
X = df[features]
y = df[target]

#80/20 train test split for non sampled data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_original = X.copy()
y_original = y.copy()

X_train_original, X_test_original, y_train_original, y_test_original = train_test_split(X_original, y_original, test_size=0.2, random_state=42)

X_lhs = df_lhs[features]
y_lhs = df_lhs[target]
#80/20 train test split for LHS samples
X_train_lhs, X_test_lhs, y_train_lhs, y_test_lhs = train_test_split(X_lhs, y_lhs, test_size=0.2, random_state=42)

X_multi_stratified = df_multi_stratified[features]
y_multi_stratified = df_multi_stratified[target]
#80/20 train test split for multi-stratified samples
X_train_multi_stratified, X_test_multi_stratified, y_train_multi_stratified, y_test_multi_stratified = train_test_split(X_multi_stratified, y_multi_stratified, test_size=0.2, random_state=42)

X_random = df_random_samples[features]
y_random = df_random_samples[target]
#80/20 train test split for random samples
X_train_random, X_test_random, y_train_random, y_test_random = train_test_split(X_random, y_random, test_size=0.2, random_state=42)

In [12]:
#Preproccessing and Model Pipeline
from pyexpat import model


numeric_features = features

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_features)
    ]
)

#standared architecture
model_standard = Pipeline([
    ("preprocessor", preprocessor),
    ("nn", MLPRegressor(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1000,
        random_state=42
    ))
])

#Narrow and Deep Model
model_narrow_deep = Pipeline([
    ("preprocessor", preprocessor),
    ("nn", MLPRegressor(
        #we'll implement a narrow and deep architecture
        hidden_layer_sizes=(256, 128, 64, 32),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1000,
        random_state=42
    ))
])

#Narrow and Shallow Model
model_narrow_shallow = Pipeline([
    ("preprocessor", preprocessor),
    ("nn", MLPRegressor(
        #we'll implement a narrow and shallow architecture
        hidden_layer_sizes=(64,),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1000,
        random_state=42
    ))
])

#Wide and Deep Model
model_wide_deep = Pipeline([
    ("preprocessor", preprocessor),
    ("nn", MLPRegressor(
        hidden_layer_sizes=(256, 128, 64, 32),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1000,
        random_state=42
    ))
])

#Narrow and Shallow
model_wide_shallow = Pipeline([
    ("preprocessor", preprocessor),
    ("nn", MLPRegressor(
        hidden_layer_sizes=(32),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1000,
        random_state=42
    ))
])

#Goofy Model #1
model_Goofy1 = Pipeline([
    ("preprocessor", preprocessor),
    ("nn", MLPRegressor(
        hidden_layer_sizes=(256, 2, 256, 3),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1000,
        random_state=42
    ))
])

#Goofy Model #2
model_Goofy2 = Pipeline([
    ("preprocessor", preprocessor),
    ("nn", MLPRegressor(
        hidden_layer_sizes=(16, 15, 14, 13, 12, 11, 10, 9, 8, 7 ,6, 5, 4),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1000,
        random_state=42
    ))
])

model_Goofy3 = Pipeline([
    ("preprocessor", preprocessor),
    ("nn", MLPRegressor(
        hidden_layer_sizes=(3, 5, 7, 11, 13, 17),
        activation="relu",
        solver="adam",
        alpha=0.001,
        learning_rate_init=0.001,
        max_iter=1000,
        random_state=42
    ))
])

models = {
    "Standard": model_standard,
    "Narrow_Deep": model_narrow_deep,
    "Narrow_Shallow": model_narrow_shallow,
    "Wide_Deep": model_wide_deep,
    "Wide_Shallow": model_wide_shallow,
    "Goofy1": model_Goofy1,
    "Goofy2": model_Goofy2,
    "Goofy3": model_Goofy3
}
datasets = {
    "Non_Sampled": (X_train, y_train),
    "LHS": (X_train_lhs, y_train_lhs),
    "Multi_Stratified": (X_train_multi_stratified, y_train_multi_stratified),
    "Random": (X_train_random, y_train_random)
}

for model_name, model in models.items():
    for dataset_name, (X_train, y_train) in datasets.items():
        print(f"Training {model_name} on {dataset_name} dataset...")
        model.fit(X_train, y_train)
        print(f"{model_name} trained on {dataset_name} dataset successfully!")


Training Standard on Non_Sampled dataset...


d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Standard trained on Non_Sampled dataset successfully!
Training Standard on LHS dataset...


d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Standard trained on LHS dataset successfully!
Training Standard on Multi_Stratified dataset...


...

d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Wide_Shallow trained on Random dataset successfully!
Training Goofy1 on Non_Sampled dataset...
Goofy1 trained on Non_Sampled dataset successfully!
Training Goofy1 on LHS dataset...
Goofy1 trained on LHS dataset successfully!
Training Goofy1 on Multi_Stratified dataset...
Goofy1 trained on Multi_Stratified dataset successfully!
Training Goofy1 on Random dataset...
Goofy1 trained on Random dataset successfully!
Training Goofy2 on Non_Sampled dataset...
Goofy2 trained on Non_Sampled dataset successfully!
Training Goofy2 on LHS dataset...


d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


Goofy2 trained on LHS dataset successfully!
Training Goofy2 on Multi_Stratified dataset...
Goofy2 trained on Multi_Stratified dataset successfully!
Training Goofy2 on Random dataset...
Goofy2 trained on Random dataset successfully!
Training Goofy3 on Non_Sampled dataset...
Goofy3 trained on Non_Sampled dataset successfully!
Training Goofy3 on LHS dataset...
Goofy3 trained on LHS dataset successfully!
Training Goofy3 on Multi_Stratified dataset...
Goofy3 trained on Multi_Stratified dataset successfully!
Training Goofy3 on Random dataset...
Goofy3 trained on Random dataset successfully!


d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


In [13]:
#predict for all fitted models across sampling strategies
predictions = {}
for model_name, model in models.items():
    for dataset_name, (X_train, y_train) in datasets.items():
        if dataset_name == "LHS":
            X_test_current = X_test_lhs
            y_test_current = y_test_lhs
        elif dataset_name == "Multi_Stratified":
            X_test_current = X_test_multi_stratified
            y_test_current = y_test_multi_stratified
        elif dataset_name == "Random":
            X_test_current = X_test_random
            y_test_current = y_test_random
        else:  # Non_Sampled
            X_test_current = X_test
            y_test_current = y_test
        
        print(f"Predicting with {model_name} on {dataset_name} test set...")
        y_pred = model.predict(X_test_current)
        predictions[(model_name, dataset_name)] = (y_pred, y_test_current)
        print(f"Prediction with {model_name} on {dataset_name} test set successful!")


Predicting with Standard on Non_Sampled test set...
Prediction with Standard on Non_Sampled test set successful!
Predicting with Standard on LHS test set...
Prediction with Standard on LHS test set successful!
Predicting with Standard on Multi_Stratified test set...
Prediction with Standard on Multi_Stratified test set successful!
Predicting with Standard on Random test set...
Prediction with Standard on Random test set successful!
Predicting with Narrow_Deep on Non_Sampled test set...
Prediction with Narrow_Deep on Non_Sampled test set successful!
Predicting with Narrow_Deep on LHS test set...
Prediction with Narrow_Deep on LHS test set successful!
Predicting with Narrow_Deep on Multi_Stratified test set...
Prediction with Narrow_Deep on Multi_Stratified test set successful!
Predicting with Narrow_Deep on Random test set...
Prediction with Narrow_Deep on Random test set successful!
Predicting with Narrow_Shallow on Non_Sampled test set...
Prediction with Narrow_Shallow on Non_Sampled 

---------------------------------------------------------------------------------------

In [14]:
from sklearn.model_selection import cross_val_score
import numpy as np

sampling_strategies = ['random', 'lhs', 'multi_stratified', 'original']
architectures = ['standard', 'wide_deep', 'narrow_deep', 'wide_shallow', 'narrow_shallow', 'Goofy1', 'Goofy2', 'Goofy3']
k = 5
kf = KFold(n_splits=k, shuffle=True, random_state=42)

results = {}
for strategy in sampling_strategies:
    X_data = globals()[f'X_{strategy}']
    y_data = globals()[f'y_{strategy}']
    for arch in architectures:
        model = globals()[f'model_{arch}']
        scores = cross_val_score(model, X_data, y_data, cv=kf, scoring='r2')
        mean_score = np.mean(scores)
        std_score = np.std(scores)
        results[(strategy, arch)] = {'mean_r2': mean_score, 'std_r2': std_score, 'fold_scores': scores}
        print(f"{strategy} + {arch}: Mean R² = {mean_score:.3f}, Std = {std_score:.3f}")

d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maxi

random + standard: Mean R² = -0.044, Std = 0.035


d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maxi

random + wide_deep: Mean R² = -0.591, Std = 0.124


d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maxi

...

original + wide_shallow: Mean R² = 0.769, Std = 0.025


d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maxi

original + narrow_shallow: Mean R² = 0.805, Std = 0.020
original + Goofy1: Mean R² = 0.968, Std = 0.006
original + Goofy2: Mean R² = 0.964, Std = 0.010


d:\Python\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


original + Goofy3: Mean R² = 0.872, Std = 0.024


In [16]:
print("\nCross-Validation Results:")
for (strategy, arch), metrics in results.items():
    print(f"{strategy} + {arch}: Mean R² = {metrics['mean_r2']:.3f}, Std = {metrics['std_r2']:.3f}, Fold Scores = {metrics['fold_scores']}")




Cross-Validation Results:
random + standard: Mean R² = -0.044, Std = 0.035, Fold Scores = [ 0.01702399 -0.06702267 -0.02892398 -0.06459953 -0.07844016]
random + wide_deep: Mean R² = -0.591, Std = 0.124, Fold Scores = [-0.59138339 -0.81924462 -0.54146104 -0.4484582  -0.55360271]
random + narrow_deep: Mean R² = -0.591, Std = 0.124, Fold Scores = [-0.59138339 -0.81924462 -0.54146104 -0.4484582  -0.55360271]
random + wide_shallow: Mean R² = -2.180, Std = 0.309, Fold Scores = [-1.89246106 -2.60537173 -1.7971735  -2.16513384 -2.44102296]
random + narrow_shallow: Mean R² = -1.433, Std = 0.240, Fold Scores = [-1.1976187  -1.75733088 -1.13680869 -1.43853478 -1.63452876]
random + Goofy1: Mean R² = -0.043, Std = 0.035, Fold Scores = [ 0.02037323 -0.06196338 -0.03210954 -0.06988282 -0.07367154]
random + Goofy2: Mean R² = -0.128, Std = 0.067, Fold Scores = [-0.17680095 -0.07363034 -0.08832319 -0.23574264 -0.06514493]
random + Goofy3: Mean R² = -0.017, Std = 0.015, Fold Scores = [ 0.00730274 -0.028